In [1]:
from backtesting import Backtest,Strategy
from backtesting.lib import crossover
import talib
import yfinance as yf
import pandas as pd
import numpy as np
import math

C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
cl = yf.Ticker('CL=F')
data = cl.history('max')

In [3]:
class MARSI(Strategy):
    ma_window = 20
    rsi_window = 14
    upper_bound = 70
    lower_bound = 30

    def init(self):
        self.ma = self.I(talib.MA, self.data.Close, self.ma_window)
        self.rsi = self.I(talib.RSI, self.data.Close, self.rsi_window)

    def next(self):
        price = self.data.Close[-1]
        m1 = self.ma[-1]
        m2 = self.ma[-2]
        curr_rsi = self.rsi[-1]
        pervious_rsi = self.rsi[-2]
        slope_rsi = curr_rsi - pervious_rsi
        slope_pct = (self.ma[-1] - self.ma[-2]) / self.ma[-2]
        slope_threshold = math.tan(math.pi/9)
        slope_angle = math.atan(slope_pct * 100)

        # ENTRY: Only enter if we don't have a position
        if not self.position:
            # Added RSI rising check and cleaned up the slope check
            if slope_angle >= slope_threshold and price > m1 and slope_rsi:
                self.buy()

        # EXIT: Give the trade more 'breathing room'
        elif self.position:
            # 1. Trailing Stop: Update SL to the current MA value if it's higher than the old SL
            # 2. Relaxed Exit: Only exit if price closes significantly below MA or RSI is pinned at top
            if price < m1 * 0.98 or curr_rsi > 85: 
                self.position.close()

In [4]:
bt = Backtest(data, MARSI, cash=10_000)

In [5]:
stats = bt.optimize(
    ma_window = range(20,50,2),
    upper_bound = range(60,80,2),
    lower_bound = range(20,40,2),
    maximize = 'Equity Final [$]',
    max_tries = 150
)

C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\backtesting.py:1624: RuntimeWarning: If you want to use multi-process optimization with `multiprocessing.get_start_method() == 'spawn'` (e.g. on Windows),set `backtesting.Pool = multiprocessing.Pool` (or of the desired context) and hide `bt.optimize()` call behind a `if __name__ == '__main__'` guard. Currently using thread-based paralellism, which might be slightly slower for non-numpy / non-GIL-releasing code. See https://github.com/kernc/backtesting.py/issues/1256
  output = _optimize_grid()


Backtest.optimize:   0%|          | 0/160 [00:00<?, ?it/s]

Backtest.run:   0%|          | 0/6556 [00:00<?, ?bar/s]

C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)


Backtest.run:   0%|          | 0/6556 [00:00<?, ?bar/s]

C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)


... (214 outputs left out)

Backtest.run:   0%|          | 0/6548 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/6538 [00:00<?, ?bar/s]

C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\backtesting.py:1637: UserWarning: Some trades remain open at the end o

Backtest.run:   0%|          | 0/6548 [00:00<?, ?bar/s]

In [6]:
stats

Start                     2000-08-23 00:00...
End                       2026-10-30 00:00...
Duration                   9564 days 00:00:00
Exposure Time [%]                    32.90754
Equity Final [$]                  31418.75094
Equity Peak [$]                   49361.36982
Return [%]                          214.18751
Buy & Hold Return [%]               223.85691
Return (Ann.) [%]                     4.48474
Volatility (Ann.) [%]                21.62296
CAGR [%]                              3.06242
Sharpe Ratio                          0.20741
Sortino Ratio                         0.31495
Calmar Ratio                          0.10071
Alpha [%]                                 NaN
Beta                                      NaN
Max. Drawdown [%]                   -44.52978
Avg. Drawdown [%]                    -8.62761
Max. Drawdown Duration     4073 days 00:00:00
Avg. Drawdown Duration      201 days 00:00:00
# Trades                                   80
Win Rate [%]                      

In [7]:
bt.plot()

GridPlot(id='p1396', ...)

In [8]:
stats['_strategy']

<Strategy MARSI(ma_window=28,upper_bound=60,lower_bound=22)>

In [9]:
class MARSI_FINAL(Strategy):
    ma_window = 28
    rsi_window = 14

    def init(self):
        self.ma = self.I(talib.MA, self.data.Close, self.ma_window)
        self.rsi = self.I(talib.RSI, self.data.Close, self.rsi_window)

    def next(self):
        price = self.data.Close[-1]
        m1 = self.ma[-1]
        m2 = self.ma[-2]
        curr_rsi = self.rsi[-1]
        pervious_rsi = self.rsi[-2]
        slope_rsi = curr_rsi - pervious_rsi
        slope_pct = (self.ma[-1] - self.ma[-2]) / self.ma[-2]
        slope_threshold = math.tan(math.pi/9)
        slope_angle = math.atan(slope_pct * 100)

        # ENTRY: Only enter if we don't have a position
        if not self.position:
            # Added RSI rising check and cleaned up the slope check
            if slope_angle >= slope_threshold and price > m1 and slope_rsi:
                self.buy()

        # EXIT: Give the trade more 'breathing room'
        elif self.position:
            # 1. Trailing Stop: Update SL to the current MA value if it's higher than the old SL
            # 2. Relaxed Exit: Only exit if price closes significantly below MA or RSI is pinned at top
            if price < m1 * 0.98 or curr_rsi > 85: 
                self.position.close()

In [10]:
data_1 = cl.history('1y')
bt_1 = Backtest(data_1, MARSI_FINAL, cash = 10_000)

In [11]:
stats1 = bt_1.run()

Backtest.run:   0%|          | 0/222 [00:00<?, ?bar/s]

In [12]:
stats1

Start                     2025-11-03 00:00...
End                       2026-10-30 00:00...
Duration                    360 days 23:00:00
Exposure Time [%]                        41.6
Equity Final [$]                  12925.01991
Equity Peak [$]                   20285.29945
Return [%]                            29.2502
Buy & Hold Return [%]                 80.7639
Return (Ann.) [%]                    29.51578
Volatility (Ann.) [%]                56.94199
CAGR [%]                             19.61751
Sharpe Ratio                          0.51835
Sortino Ratio                         1.02715
Calmar Ratio                          0.70572
Alpha [%]                           -16.30585
Beta                                  0.56406
Max. Drawdown [%]                   -41.82354
Avg. Drawdown [%]                   -10.77912
Max. Drawdown Duration      206 days 00:00:00
Avg. Drawdown Duration       43 days 00:00:00
# Trades                                    6
Win Rate [%]                      

In [13]:
bt_1.plot()

GridPlot(id='p1816', ...)